In [1]:
from cellm import ask
from cellm.core import chat

In [2]:
ask("How is the `ask` function defined")

##### 🤖 LLM Response (glm-5.3-flash | in: 1425 | out: 273)

> `ask` is defined in `src/cellm/core.py` (lines 62–72):
> 
> ```python
> def ask(pr: str):
>     _, cells_raw, active_cell_id = get_cells()
>     cells_content = json.dumps(cells_raw)
>     chat.hist = [f"active notebook content: {cells_content}"]
>     resp = chat(pr)
>     chat.last_response = resp
> 
>     model_name = chat.model.split("/")[-1]
>     header = f"##### 🤖 LLM Response ({model_name} | in: {resp.usage.prompt_tokens} | out: {resp.usage.completion_tokens})\n\n> "
>     quoted = resp.choices[0].message.content.replace("\n", "\n> ")
>     insert_cell(header + quoted, active_cell_id)
> ```
> 
> It fetches notebook cells from the extension (`get_cells()` via `http://localhost:3144`), resets `chat.hist` to just the serialized cells + question, calls the `Chat` (lisette) instance, then inserts a blockquoted, header-annotated response below the active cell.

Q: Can you check the definition of `chat`? Are there any setting/parameters im not taking advantage of yet?

In [4]:
ask("")

##### 🤖 LLM Response (glm-5.3-flash | in: 737 | out: 164)

> Not yet complete. The tree confirms the package is at `src/cellm/` (`core.py`, `helpers.py`, `tools.py`). My grep for `class Chat|def __init__` under `src/cellm` returned "Not Found" (possibly a pattern quirk), so next step is to view `src/cellm/core.py` directly to inspect the `chat` instance/`Chat` class and enumerate unused settings/parameters (e.g., temperature, system prompt, tools, history handling).

In [5]:
ask("ok, the package is installed in the .venv, you can find it there!")

##### 🤖 LLM Response (glm-5.3-flash | in: 5268 | out: 553)

> `chat` is a **lisette `Chat`** instance (`.venv/.../lisette/core.py:523`), created at `src/cellm/core.py:53`. Full signature (line 524–547):
> 
> ```python
> Chat(model, sp='', temp=0, search=False, tools=None, hist=None,
>      ns=None, cache=False, cache_idxs=[-1], ttl=None,
>      api_base=None, api_key=None, extra_headers=None,
>      tc_refs=False, tc_res_eval=False, markup=0, tool_reminder=None,
>      max_tokens=None, completefunc=None, stream=False, callkw=None)
> ```
> 
> Notable settings you're likely not using:
> - **`sp`** – system prompt
> - **`tools`/`ns`** – tool calling with a custom namespace
> - **`search`** – web search (`'l'/'m'/'h'`) per-call or at init; also overridable per call
> - **`think`** (per-call kwarg, not init) – reasoning effort `'l'/'m'/'h'`/`'x'` (`_think_kw`, line 603)
> - **`prefill`** (per-call) – assistant prefill
> - **`stream`** – streaming with usage tracking
> - **`cache`/`cache_idxs`/`ttl`** – Anthropic prompt caching
> - **`temp`** – currently 0; per-call override supported
> - **`max_tokens`**, **`max_steps`** (tool-loop cap), **`final_prompt`**, **`return_all`**, **`tool_choice`**
> - **`tc_refs`/`tc_res_eval`** – tool call result reuse
> - **`markup`**, **`tool_reminder`**, **`callkw`** – cost markup, transient tool reminder, default call kwargs
> - `chat.use` accumulates `UsageStats` (cost/tokens) you're not surfacing
> 
> **Remaining work:** view `src/cellm/core.py` (esp. lines ~50–60) to see exactly which of these args your `chat = Chat(...)` already passes, so I can diff against the list above and give you a precise "unused" set.